# Supplimentary material stuff

In [1]:
import os
import glob
import networkx as nx
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
BASE_DIR = (r"C:\Users\keevo\OneDrive\Documents\Keevon Jacobs Wits"
            r"\Honors year\Research Program\Experiments\5.Directed_Ground_Truths")

GT_RANGE = list(range(1, 11))

# ----------------------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------------------
def load_graph(i):
    """Finds and loads the graphml file for ground truth i."""
    # Pattern handles Directed_GT_1000, Directed_GT_1000.graphml, etc.
    pattern = os.path.join(BASE_DIR, f"Directed_GT_{i}000*")
    matches = glob.glob(pattern)
    
    # Filter out directories, just in case
    matches = [m for m in matches if os.path.isfile(m)]
    
    if not matches:
        print(f"Warning: Could not find file for GT{i} using pattern {pattern}")
        return None
        
    path = matches[0]
    try:
        G = nx.read_graphml(path)
        return nx.DiGraph(G)
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

def calculate_directed_shd(G1, G2):
    """
    Calculates standard Directed SHD (Added + Deleted + Reversed).
    """
    edges1 = set(G1.edges())
    edges2 = set(G2.edges())
    
    # Reversed edges: u->v in G1 and v->u in G2
    reversed_edges = {(u, v) for (u, v) in edges1 if (v, u) in edges2}
    
    # Missing: in G1 but not in G2 (includes reversed)
    missing = edges1 - edges2
    
    # Added: in G2 but not in G1 (includes reversed)
    added = edges2 - edges1
    
    # SHD = missing + added - reversed (to not double count)
    return len(missing) + len(added) - len(reversed_edges)

def print_terminal_heatmap(df):
    """
    Prints a pandas DataFrame as a color-coded heatmap in the terminal.
    Green = low value, Red = high value.
    """
    vals = df.values
    vmin, vmax = np.min(vals), np.max(vals)
    
    # Avoid division by zero if all values are the same
    if vmax == vmin:
        vmax += 1 
        
    print("\n=== Directed SHD Heatmap (GT vs GT) ===")
    
    # Print column headers
    header = "      " + "".join([f"GT{j:<4}" for j in df.columns])
    print(header)
    print("-" * len(header))
    
    for i, row_idx in enumerate(df.index):
        row_str = f"GT{row_idx:<4} "
        for j, val in enumerate(df.loc[row_idx]):
            # Normalize value between 0 and 1
            norm_val = (val - vmin) / (vmax - vmin)
            
            # Map to ANSI 256 color code (green to yellow to red)
            # 46 is Green, 226 is Yellow, 196 is Red
            if norm_val < 0.5:
                # Green to Yellow
                color_code = int(46 + (226 - 46) * (norm_val / 0.5))
            else:
                # Yellow to Red
                color_code = int(226 + (196 - 226) * ((norm_val - 0.5) / 0.5))
                
            # Print with background color and black text
            row_str += f"\033[48;5;{color_code}m {val:4d} \033[0m "
        print(row_str)
    print("=" * len(header) + "\n")

# ----------------------------------------------------------------------
# Main Execution
# ----------------------------------------------------------------------
graphs = {}
print("Loading graphs...")
for i in GT_RANGE:
    G = load_graph(i)
    if G is not None:
        graphs[i] = G
        print(f"  Loaded GT{i}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Initialize a 10x10 matrix with zeros
matrix = pd.DataFrame(0, index=list(graphs.keys()), columns=list(graphs.keys()))

print("\nCalculating Directed SHD...")
for i in graphs:
    for j in graphs:
        if i == j:
            matrix.loc[i, j] = 0
        else:
            matrix.loc[i, j] = calculate_directed_shd(graphs[i], graphs[j])

# 1. Output to Terminal Heatmap
print_terminal_heatmap(matrix)

# 2. Save to CSV
out_csv = os.path.join(BASE_DIR, "GT_to_GT_Directed_SHD.csv")
matrix.to_csv(out_csv)
print(f"Saved raw SHD matrix to: {out_csv}")

Loading graphs...
  Loaded GT1: 37 nodes, 36 edges
  Loaded GT2: 37 nodes, 36 edges
  Loaded GT3: 37 nodes, 36 edges
  Loaded GT4: 37 nodes, 36 edges
  Loaded GT5: 37 nodes, 36 edges
  Loaded GT6: 37 nodes, 36 edges
  Loaded GT7: 37 nodes, 36 edges
  Loaded GT8: 37 nodes, 36 edges
  Loaded GT9: 37 nodes, 36 edges
  Loaded GT10: 37 nodes, 36 edges

Calculating Directed SHD...

=== Directed SHD Heatmap (GT vs GT) ===
      GT1   GT2   GT3   GT4   GT5   GT6   GT7   GT8   GT9   GT10  
------------------------------------------------------------------
GT1        0      2      0      8      9      4      8      8      6      2  
GT2        2      0      2      6      7      2      6      6      4      0  
GT3        0      2      0      8      9      4      8      8      6      2  
GT4        8      6      8      0      5      6      2      2      4      6  
GT5        9      7      9      5      0      9      5      5      7      7  
GT6        4      2      4      6      9      0      4   

In [2]:
import os
import pandas as pd

# Define the base directory
BASE_DIR = (r"C:\Users\keevo\OneDrive\Documents\Keevon Jacobs Wits"
            r"\Honors year\Research Program\Experiments\11.SHAPLY")

GT_RANGE = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
SAMPLE_SIZES = [50, 100, 500, 1000, 2000, 3000, 5000, 6000]

# Counter to track successful reads
file_count = 0

for i in GT_RANGE:
    for j in SAMPLE_SIZES:
        # Construct the path
        file_path = os.path.join(BASE_DIR, str(i), f"sample{j}", "XGBoost_Model_Performance.csv")
        
        print(f"\n{'='*60}")
        print(f"GROUND TRUTH: {i} | SAMPLE SIZE: {j}")
        print(f"{'='*60}")
        
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path)
                # Print the dataframe without the index column, formatted nicely
                print(df.to_string(index=False))
                file_count += 1
            except Exception as e:
                print(f"Error reading file: {e}")
        else:
            print(f"File not found: {file_path}")

print(f"\n\nSuccessfully printed {file_count} out of 80 files.")


GROUND TRUTH: 1 | SAMPLE SIZE: 50
      Target  Accuracy  Weighted_F1
 ANAPHYLAXIS       0.8     0.762500
      ARTCO2       0.8     0.802778
          BP       0.4     0.385714
    CATECHOL       0.6     0.525000
          CO       0.7     0.702222
         CVP       0.4     0.360000
  DISCONNECT       0.6     0.525000
   ERRCAUTER       0.9     0.901010
ERRLOWOUTPUT       0.8     0.780952
      EXPCO2       0.3     0.230769
        FIO2       0.9     0.886275
     HISTORY       0.7     0.690110
          HR       0.7     0.654545
        HRBP       0.4     0.382222
       HREKG       0.6     0.600000
       HRSAT       0.7     0.651282
 HYPOVOLEMIA       0.3     0.292929
INSUFFANESTH       0.6     0.600000
  INTUBATION       0.3     0.311111
  KINKEDTUBE       0.9     0.852632
  LVEDVOLUME       0.8     0.790000
   LVFAILURE       0.7     0.696970
      MINVOL       0.4     0.333333
   MINVOLSET       0.3     0.260000
         PAP       0.6     0.560000
        PCWP       0.6     0.